In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import json
from datetime import datetime
import time

In [0]:
def fetch_ergast_data(season, endpoint):
    """Busca dados da Ergast API com flatten completo de estruturas aninhadas"""
    url = f"https://api.jolpi.ca/ergast/f1/{season}/{endpoint}.json"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        if "MRData" in data:
            if endpoint == "drivers" and "DriverTable" in data["MRData"]:
                items = data["MRData"]["DriverTable"].get("Drivers", [])

            elif endpoint == "constructors" and "ConstructorTable" in data["MRData"]:
                items = data["MRData"]["ConstructorTable"].get("Constructors", [])

            elif endpoint == "races" and "RaceTable" in data["MRData"]:
                items = data["MRData"]["RaceTable"].get("Races", [])
                # Flatten Circuit
                for race in items:
                    if "Circuit" in race and isinstance(race["Circuit"], dict):
                        race["circuitId"] = race["Circuit"].get("circuitId")
                        del race["Circuit"]

            elif endpoint == "results" and "RaceTable" in data["MRData"]:
                items = []
                races = data["MRData"]["RaceTable"].get("Races", [])

                for race in races:
                    race_results = race.get("Results", [])

                    for result in race_results:
                        # Flatten Driver
                        if "Driver" in result and isinstance(result["Driver"], dict):
                            result["driverId"] = result["Driver"].get("driverId")
                            del result["Driver"]

                        # Flatten Constructor
                        if "Constructor" in result and isinstance(result["Constructor"], dict):
                            result["constructorId"] = result["Constructor"].get("constructorId")
                            del result["Constructor"]

                        # Flatten Status
                        if "Status" in result and isinstance(result["Status"], dict):
                            result["statusId"] = result["Status"].get("statusId")
                            del result["Status"]

                        # Flatten Time (pode ser um objeto com value)
                        if "Time" in result and isinstance(result["Time"], dict):
                            result["time"] = result["Time"].get("time")
                            result["milliseconds"] = result["Time"].get("millis")
                            del result["Time"]

                        # Flatten FastestLap (IMPORTANTE - esse é o seu erro)
                        if "FastestLap" in result and isinstance(result["FastestLap"], dict):
                            fastest_lap = result["FastestLap"]
                            result["fastestLap"] = fastest_lap.get("rank")

                            # Flatten Time dentro de FastestLap
                            if "Time" in fastest_lap and isinstance(fastest_lap["Time"], dict):
                                result["fastestLapTime"] = fastest_lap["Time"].get("time")
                            else:
                                result["fastestLapTime"] = fastest_lap.get("time")

                            result["fastestLapSpeed"] = fastest_lap.get("AverageSpeed", {}).get("speed") if isinstance(fastest_lap.get("AverageSpeed"), dict) else fastest_lap.get("AverageSpeed")
                            del result["FastestLap"]

                        result["raceId"] = race.get("raceId", race.get("round"))
                        result["_season"] = season
                        items.append(result)
            else:
                items = []

            # Adiciona metadados
            for item in items:
                item["_ingested_at"] = datetime.now().isoformat()
                item["_source"] = "ergast_api"
                if "_season" not in item:
                    item["_season"] = season

            return items
        return []

    except Exception as e:
        print(f"Erro ao buscar {endpoint}/{season}: {e}")
        import traceback
        traceback.print_exc()
        return []
def drivers_raw():
    """Pipeline DLT para ingestão de dados de pilotos"""
    all_drivers = []
    seasons = list(range(1950, 2025))

    for season in seasons[:10]:
        drivers = fetch_ergast_data(season, "drivers")
        all_drivers.extend(drivers)
        time.sleep(0.5)

    if all_drivers:
        return spark.createDataFrame(all_drivers)
    else:
        return spark.createDataFrame(
            [],
            schema=StructType([
                StructField("driverId", StringType(), True),
                StructField("driverRef", StringType(), True),
                StructField("number", StringType(), True),
                StructField("code", StringType(), True),
                StructField("forename", StringType(), True),
                StructField("surname", StringType(), True),
                StructField("nationality", StringType(), True),
                StructField("dateOfBirth", StringType(), True),
                StructField("url", StringType(), True),
                StructField("_ingested_at", StringType(), True),
                StructField("_source", StringType(), True),
                StructField("_season", IntegerType(), True)
            ])
        )
a = drivers_raw()
display(a)